# GTFS Scheduled Trips and Stops Analysis

This script processes GTFS data to analyze transit service for a selection of agencies across different days of the week. The main steps are:
1. Filter trips by agency and date: Selects scheduled trips for the specified agencies on a Sunday, Saturday, and a weekday.
2. Extract feed keys and retrieve stops: For each day, unique feed keys from the trips are used to query the corresponding daily scheduled stops, including stop identifiers, names, and associated route information.
3. Expand route-stop relationships: Many stops are served by multiple routes. The script transforms the stops data so that each row represents a unique stop × route combination.
4. Merge with trip counts: Trip-level data is aggregated by feed and route to calculate the number of trips per route. This information is merged with the expanded stops data, providing trip counts for each stop × route combination.
5. Aggregate stop-level statistics: Stops are summarized by feed, stop ID, stop name, and route type to calculate:
- Total number of trips serving each stop.
- Number of distinct routes serving each stop.

The final dataset provides a detailed overview of transit service at the stop level, including how many trips and routes serve each stop for the selected agencies and dates.

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
import datetime as dt
from calitp_data_analysis.sql import get_engine
from shared_utils import gtfs_utils_v2
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

/opt/conda/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
agencies_to_find = [
    "Bay Area 511 BART Schedule",
    "Big Blue Bus Swiftly Schedule",
    "Caltrain Schedule",
    "Culver City Schedule",
    "Foothill Schedule",
    "Fresno Schedule",
    "Gold Coast Schedule",
    "Golden Gate Park Shuttle Schedule",
    "Bay Area 511 Golden Gate Transit Schedule",
    "Long Beach Schedule",
    "OCTA Schedule",
    "OmniTrans Schedule",
    "Riverside Schedule",
    "Sacramento Schedule",
    "Santa Cruz Schedule",
    "SBMTD Schedule",
    "SamTrans Schedule",
    "San Diego Schedule"
]

agency_sql = ', '.join(f"'{agency}'" for agency in agencies_to_find)

In [5]:
trip_columns = [
    "feed_key", "trip_instance_key", "gtfs_dataset_key", "trip_id", "route_id", 
    "route_type", "name", "route_short_name", "route_long_name", "service_date", 
    "trip_first_departure_datetime_local_tz", "trip_last_arrival_datetime_local_tz",
    "time_of_day", "service_hours", "num_stop_times", "num_distinct_stops_served",
    "frequencies_defined_trip", "is_gtfs_flex_trip", "is_entirely_demand_responsive_trip"
]

In [6]:
with db_engine.connect() as connection:
    query = f"""
        SELECT {', '.join(trip_columns)}
        FROM `cal-itp-data-infra.mart_gtfs.fct_scheduled_trips`
        WHERE service_date = DATE('2025-05-18')
          AND name IN ({agency_sql})
    """
    trips_sunday_data = pd.read_sql(query, connection)



with db_engine.connect() as connection:
    query = f"""
        SELECT {', '.join(trip_columns)}
        FROM `cal-itp-data-infra.mart_gtfs.fct_scheduled_trips`
        WHERE service_date = DATE('2025-05-21')
          AND name IN ({agency_sql})
    """
    trips_weekday_data = pd.read_sql(query, connection)


with db_engine.connect() as connection:
    query = f"""
        SELECT {', '.join(trip_columns)}
        FROM `cal-itp-data-infra.mart_gtfs.fct_scheduled_trips`
        WHERE service_date = DATE('2025-05-17')
          AND name IN ({agency_sql})
    """
    trips_saturday_data = pd.read_sql(query, connection)


In [ ]:
# Get unique feed_keys from your trips_weekday_data
weekday_feed_key = trips_weekday_data['feed_key'].unique().tolist()

# Convert to SQL-friendly string for IN clause
feed_keys_str_weekday = ",".join(f"'{fk}'" for fk in weekday_feed_key)

# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2025-05-21')
          AND feed_key IN ({feed_keys_str_weekday})
    """
    stops_unique_weekday = pd.read_sql(query, connection)

In [ ]:
# Get unique feed_keys from your trips_weekday_data
saturday_feed_key = trips_saturday_data['feed_key'].unique().tolist()

# Convert to SQL-friendly string for IN clause
feed_keys_str_saturday = ",".join(f"'{fk}'" for fk in saturday_feed_key)

# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2025-05-17')
          AND feed_key IN ({feed_keys_str_saturday})
    """
    stops_unique_saturday = pd.read_sql(query, connection)

In [ ]:
# Get unique feed_keys from your trips_weekday_data
sunday_feed_key = trips_sunday_data['feed_key'].unique().tolist()

# Convert to SQL-friendly string for IN clause
feed_keys_str_sunday = ",".join(f"'{fk}'" for fk in sunday_feed_key)

# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2025-05-17')
          AND feed_key IN ({feed_keys_str_sunday})
    """
    stops_unique_sunday = pd.read_sql(query, connection)

In [ ]:
stops_unique_weekday.info()

In [ ]:
# trips per feed_key × route_id
trips_per_route = trips_weekday_data.groupby(['feed_key', 'route_id']).agg(
    n_trips=('trip_id', 'nunique')
).reset_index()

In [ ]:
def expand_routes_fixed(row):
    route_ids = row['route_id_array'] if isinstance(row['route_id_array'], list) else []
    route_types = row['route_type_array'] if isinstance(row['route_type_array'], list) else []
    
    # If there is only one type but multiple routes, repeat the type
    if len(route_types) == 1 and len(route_ids) > 1:
        route_types = route_types * len(route_ids)
    
    # If lengths match, zip safely
    pairs = list(zip(route_ids, route_types))
    
    if not pairs:
        return pd.DataFrame({
            'feed_key': [row['feed_key']],
            'stop_id': [row['stop_id']],
            'stop_name': [row['stop_name']],
            'route_id': [None],
            'route_type': [None]
        })
    
    return pd.DataFrame({
        'feed_key': [row['feed_key']] * len(pairs),
        'stop_id': [row['stop_id']] * len(pairs),
        'stop_name': [row['stop_name']] * len(pairs),
        'route_id': [r[0] for r in pairs],
        'route_type': [r[1] for r in pairs]
    })